In [ ]:
import torch
TORCH_VER = torch.__version__
gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f'torch={TORCH_VER}  cuda={torch.version.cuda}')
print(f'GPU: {gpu_name}  compute_capability={cap[0]}.{cap[1]}')
# P100 (6.0) is known to fail with MusicGen; T4 (7.5) works
if cap < (7, 0):
    raise RuntimeError(
        f'GPU {gpu_name} (compute {cap[0]}.{cap[1]}) is too old. '        f'Select GPU T4 x2 in Kaggle Settings.'
    )

In [ ]:
!pip install -q transformers soundfile accelerate 'huggingface-hub>=1.5.0' torch=={TORCH_VER}

In [ ]:
import csv
import os
from pathlib import Path
from tqdm import tqdm
from transformers import pipeline
import soundfile as sf
import shutil

In [ ]:
synthesiser = pipeline("text-to-audio", "csc-unipd/tasty-musicgen-small")

In [ ]:
import os
print('Dataset input directory tree:')
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = '  ' * (level + 1)
    for f in files:
        print(f'{sub_indent}{f}')

In [ ]:
INPUT_DIR = Path("/kaggle/input/datasets/mfreyeso/ablation-reprompts")
OUTPUT_DIR = Path("/kaggle/working")

# Search recursively — Kaggle may nest files in subdirectories
csv_files = sorted(INPUT_DIR.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {f}")

for csv_file in csv_files:
    # use CSV stem as subdirectory name
    sub_dir = OUTPUT_DIR / csv_file.stem
    sub_dir.mkdir(parents=True, exist_ok=True)
    print(f"\nProcessing: {csv_file.name} -> {sub_dir.name}/")

    with open(csv_file, "r") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    for row in tqdm(rows, desc=csv_file.stem):
        id_prompt = row["id_prompt"]
        reprompt = row["reprompt"]
        output_path = sub_dir / f"{id_prompt}.wav"

        if output_path.exists():
            print(f"  Skipping {id_prompt} (exists)")
            continue

        music = synthesiser(reprompt, forward_params={"do_sample": True})
        sf.write(str(output_path), music["audio"].squeeze(), music["sampling_rate"])

    # zip each subdirectory for easier download
    archive = OUTPUT_DIR / csv_file.stem
    shutil.make_archive(str(archive), "zip", str(sub_dir))
    print(f"  ✓ Archived: {csv_file.stem}.zip")


In [ ]:
wav_count = len(list(OUTPUT_DIR.rglob("*.wav")))
zip_count = len(list(OUTPUT_DIR.glob("*.zip")))
print(f"\nDone! {wav_count} WAV files in {zip_count} archives")